In [75]:
import pandas as pd
import seaborn as sns
import xgboost as xgb
import numpy as np

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

"""
---------------------------------------------------------------------------------------------------------------
Implementação de ensemble baseado em Blending dos modelos EVI, MMAING e EARS com um meta-learner XGBoost.
---------------------------------------------------------------------------------------------------------------
Metodologia: 
O Stacking, ou generalização empilhada, é uma técnica de ensemble que combina múltiplos modelos para gerar uma nova predição. 
A estratégia é usar as predições de vários modelos base (chamados de "nível 0") como entrada para um novo modelo, conhecido como meta-modelo ou blender ("nível 1"). 
Este blender aprende a melhor forma de combinar as predições dos modelos base para obter um resultado final mais acurado.

Modelos Base (Nível 0): EARS, EVI, .....

Meta-Modelo (Nível 1): XGBClassifier.

---------------------------------------------------------------------------------------------------------------
Princípios de Funcionamento:
O xgb.XGBClassifier é uma implementação otimizada do algoritmo de Gradient Boosting (Aumento de Gradiente), que funciona combinando uma série de modelos de previsão fracos, 
geralmente árvores de decisão, de forma sequencial para criar um modelo forte e altamente preciso para tarefas de classificação.

O funcionamento do XGBClassifier baseia-se em alguns conceitos-chave: 
- Ensemble Learning (Aprendizado em Conjunto): Em vez de usar um único modelo complexo, ele treina múltiplos modelos mais simples (weak learners) e agrega seus resultados.

- Gradient Boosting: Os modelos são treinados de forma iterativa. Cada nova árvore de decisão tenta corrigir os erros (resíduos) cometidos pelas árvores anteriores. 
O algoritmo usa técnicas de otimização baseadas em gradiente (descida do gradiente) para minimizar uma função de perda (erro) em cada passo.

- Árvores de Decisão: Cada "aprendiz fraco" no ensemble é uma árvore de decisão. O algoritmo encontra os melhores pontos de divisão nos dados para reduzir o erro o máximo possível.

---------------------------------------------------------------------------------------------------------------
Autor:
Andrêza Leite de Alencar, PhD
Federal Rural University of Pernambuco (UFRPE)
Centre for Data and Knowledge Integration for Health (CIDACS/FIOCRUZ)

---------------------------------------------------------------------------------------------------------------
Data: 
10/11/2025

---------------------------------------------------------------------------------------------------------------

"""

'\n---------------------------------------------------------------------------------------------------------------\nImplementação de ensemble baseado em Blending dos modelos EVI, MMAING e EARS com um meta-learner XGBoost.\n---------------------------------------------------------------------------------------------------------------\nMetodologia: \nO Stacking, ou generalização empilhada, é uma técnica de ensemble que combina múltiplos modelos para gerar uma nova predição. \nA estratégia é usar as predições de vários modelos base (chamados de "nível 0") como entrada para um novo modelo, conhecido como meta-modelo ou blender ("nível 1"). \nEste blender aprende a melhor forma de combinar as predições dos modelos base para obter um resultado final mais acurado.\n\nModelos Base (Nível 0): EARS, EVI, .....\n\nMeta-Modelo (Nível 1): XGBClassifier.\n\n---------------------------------------------------------------------------------------------------------------\nPrincípios de Funcionamento:\nO

# --- 1. Carregamento dos Dados ---

In [76]:
try:
    #df = pd.read_parquet('data_ens_1_11_25.parquet')
    #df = pd.read_parquet('aesop_with_MEM_26_03_2026.parquet')
    #df = pd.read_parquet('dado.parquet')
    #df = pd.read_parquet('aesop_26_03_2026_with_MEM_EVI.parquet')
    #df = pd.read_parquet('aesop_31_03_2026_with_MEM_all_models.parquet')
    df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/aesop_06_05_2026_with_MEM_all_mods_ens.parquet')



    
except FileNotFoundError:
    print("Arquivo não encontrado.")
    
    df = pd.DataFrame(data)

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 788655 entries, 0 to 788654
Data columns (total 40 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   co_ibge                           788655 non-null  int32  
 1   epiweek                           788655 non-null  int32  
 2   year                              788655 non-null  float64
 3   year_week                         788655 non-null  object 
 4   week                              788655 non-null  object 
 5   atend_ivas                        788655 non-null  int32  
 6   atend_totais                      788655 non-null  int32  
 7   mem_surge_01_correct_with_consec  788655 non-null  int32  
 8   warning_final_mem_surge_01        788655 non-null  int32  
 9   sinal_ears_atend                  788655 non-null  int64  
 10  sinal_evi_ivas                    788655 non-null  int64  
 11  EWS_ISF                           788655 non-null  int64 

In [78]:
df.columns.tolist()

['co_ibge',
 'epiweek',
 'year',
 'year_week',
 'week',
 'atend_ivas',
 'atend_totais',
 'mem_surge_01_correct_with_consec',
 'warning_final_mem_surge_01',
 'sinal_ears_atend',
 'sinal_evi_ivas',
 'EWS_ISF',
 'EWS_LOF',
 'EWS_OCSVM',
 'EWS_COPOD',
 'EWS_Rt',
 'hard_voting_ivas',
 'sinal_ears_atend_lag_1',
 'sinal_ears_atend_lag_2',
 'sinal_ears_atend_lag_3',
 'sinal_evi_ivas_lag_1',
 'sinal_evi_ivas_lag_2',
 'sinal_evi_ivas_lag_3',
 'EWS_ISF_lag_1',
 'EWS_ISF_lag_2',
 'EWS_ISF_lag_3',
 'EWS_LOF_lag_1',
 'EWS_LOF_lag_2',
 'EWS_LOF_lag_3',
 'EWS_OCSVM_lag_1',
 'EWS_OCSVM_lag_2',
 'EWS_OCSVM_lag_3',
 'EWS_COPOD_lag_1',
 'EWS_COPOD_lag_2',
 'EWS_COPOD_lag_3',
 'EWS_Rt_lag_1',
 'EWS_Rt_lag_2',
 'EWS_Rt_lag_3',
 'risk_probs',
 'sinal_ens_ivas_new']

# --- 2. Preparação dos Dados ---


In [79]:
#=====================================================================
#!!!!!!!  ajustar com o dado final !!!!!!! 
#=====================================================================

columns_sinais = [
    #'farrrignton', #não esta no arquivo
    'sinal_ears_atend',
     'sinal_evi_ivas',
     'EWS_ISF',
     'EWS_LOF',
     'EWS_OCSVM',
     'EWS_COPOD',
     'EWS_Rt' #NGM
] 




## --- 2.1 Tratamento de NANs ---

In [80]:
print(df[columns_sinais].isna().sum())
#tratamento
for col in columns_sinais:
    print(f"Valores NaN: '{df[col].isna().sum()}' na coluna '{col}' preenchidos com a moda")
    if df[col].isna().any():
        moda = df[col].mode()[0] 
        df[col].fillna(moda, inplace=True)
        print(f"Valores NaN na coluna '{col}' preenchidos com a moda: {int(moda)}")


print("\nVerificando dados faltantes (NaN) após o tratamento:")
print(df[columns_sinais].isnull().sum())

sinal_ears_atend    0
sinal_evi_ivas      0
EWS_ISF             0
EWS_LOF             0
EWS_OCSVM           0
EWS_COPOD           0
EWS_Rt              0
dtype: int64
Valores NaN: '0' na coluna 'sinal_ears_atend' preenchidos com a moda
Valores NaN: '0' na coluna 'sinal_evi_ivas' preenchidos com a moda
Valores NaN: '0' na coluna 'EWS_ISF' preenchidos com a moda
Valores NaN: '0' na coluna 'EWS_LOF' preenchidos com a moda
Valores NaN: '0' na coluna 'EWS_OCSVM' preenchidos com a moda
Valores NaN: '0' na coluna 'EWS_COPOD' preenchidos com a moda
Valores NaN: '0' na coluna 'EWS_Rt' preenchidos com a moda

Verificando dados faltantes (NaN) após o tratamento:
sinal_ears_atend    0
sinal_evi_ivas      0
EWS_ISF             0
EWS_LOF             0
EWS_OCSVM           0
EWS_COPOD           0
EWS_Rt              0
dtype: int64


In [81]:
df.head()

,co_ibge,epiweek,year,year_week,week,atend_ivas,atend_totais,mem_surge_01_correct_with_consec,warning_final_mem_surge_01,sinal_ears_atend,...,EWS_OCSVM_lag_2,EWS_OCSVM_lag_3,EWS_COPOD_lag_1,EWS_COPOD_lag_2,EWS_COPOD_lag_3,EWS_Rt_lag_1,EWS_Rt_lag_2,EWS_Rt_lag_3,risk_probs,sinal_ens_ivas_new
0,110001,42,2022.0,2022-42,2022-10-23,31,648,0,0,1,...,0,0,0,0,0,0,0,0,0.465267,0
1,110001,43,2022.0,2022-43,2022-10-30,16,650,0,0,0,...,0,0,0,0,0,1,0,0,0.427623,0
2,110001,44,2022.0,2022-44,2022-11-06,20,558,0,0,0,...,1,0,0,0,0,0,1,0,0.181752,0
3,110001,45,2022.0,2022-45,2022-11-13,32,677,0,0,1,...,0,1,0,0,0,0,0,1,0.506711,1
4,110001,46,2022.0,2022-46,2022-11-20,47,808,0,0,1,...,0,0,0,0,0,1,0,0,0.891895,1


## --- 2.2 Função para criar Lag Features (Janelas de Atraso) ---


In [82]:
# --- Função para criar Lag Features (Janelas de Atraso) ---
def criar_lag_features(df, colunas_sinais, lags=[1, 2, 3]):
    """
    Cria colunas de atraso (lags) para dar contexto histórico ao modelo.
    Ex: Se lag=1, cria uma coluna com o valor da semana passada.
    """
    df_lagged = df.copy()
    
    # Ordena por tempo ('year_week') e local ('co_ibge7') para garantir que o shift pegue a semana anterior correta
    if 'co_ibge7' in df.columns:
        df_lagged = df_lagged.sort_values(by=['co_ibge7', 'year_week'])
    else:
        df_lagged = df_lagged.sort_values(by=['year_week'])

    new_features = []
    
    for col in colunas_sinais:
        for lag in lags:
            nome_nova_coluna = f"{col}_lag{lag}"
            # O shift(lag) empurra os valores para baixo
            if 'co_ibge7' in df.columns:
                # Agrupa por município para não pegar dado de uma cidade e jogar na outra
                df_lagged[nome_nova_coluna] = df_lagged.groupby('co_ibge7')[col].shift(lag)
            else:
                df_lagged[nome_nova_coluna] = df_lagged[col].shift(lag)
            
            new_features.append(nome_nova_coluna)
    
    # Trata as primeiras linhas que ficaram vazias (NaN) por causa do shift
    for col in new_features:
        print(f"QTD Valores NaN na coluna '{col}' : '{df_lagged[col].isnull().sum()}' ")
        if df_lagged[col].isnull().any():
            moda = df_lagged[col].mode()[0] # .mode() retorna uma série, pegamos o primeiro valor
            df_lagged[col].fillna(moda, inplace=True)
            print(f"Valores NaN na coluna '{col}' preenchidos com a moda: '{int(moda)}' ")
    #df_lagged = df_lagged.dropna()

    
    return df_lagged, new_features



## --- 2.2.1 Aplicação no DataFrame ---


In [83]:
# --- Aplicação no DataFrame ---

# Cria os lags de 1, 2 e 3 semanas atrás
df_com_lags, nomes_features_lags = criar_lag_features(df, columns_sinais, lags=[1, 2, 3])

print(f"Novas features criadas: {nomes_features_lags}")
print(f"Tamanho do dataset original: {len(df)}")
print(f"Tamanho do dataset com lags: {len(df_com_lags)}")

# ---  Atualiza as variáveis de Treino ---
# Agora a lista de features deve incluir as originais E as novas
features_para_treino = columns_sinais + nomes_features_lags


QTD Valores NaN na coluna 'sinal_ears_atend_lag1' : '1' 
Valores NaN na coluna 'sinal_ears_atend_lag1' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'sinal_ears_atend_lag2' : '2' 
Valores NaN na coluna 'sinal_ears_atend_lag2' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'sinal_ears_atend_lag3' : '3' 
Valores NaN na coluna 'sinal_ears_atend_lag3' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'sinal_evi_ivas_lag1' : '1' 
Valores NaN na coluna 'sinal_evi_ivas_lag1' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'sinal_evi_ivas_lag2' : '2' 
Valores NaN na coluna 'sinal_evi_ivas_lag2' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'sinal_evi_ivas_lag3' : '3' 
Valores NaN na coluna 'sinal_evi_ivas_lag3' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'EWS_ISF_lag1' : '1' 
Valores NaN na coluna 'EWS_ISF_lag1' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'EWS_ISF_lag2' : '2' 
Valores NaN na coluna 'EWS_ISF_lag2' preenchidos com a mod

## --- 2.3 Configuração das features para o modelo ---

In [84]:
# Colunas dos modelos base que servirão de entrada para o modelo
df = df_com_lags #com antecipacao
base_models_columns=features_para_treino #configurado acima

#==============================================================
# variável alvo 
target_column =  'mem_surge_01_correct_with_consec' #'warning_final_mem_surge_01'
#==============================================================


## --- 2.4 Divisao dos dados  ---

In [85]:
# X são as predições dos modelos base (features para o blender)
X = df[base_models_columns]
# y é o resultado verdadeiro que queremos prever
y = df[target_column]

In [86]:
# ---  Divisão em Dados de Treino e Teste ---
# Dividimos os dados para treinar o blender e depois testar sua eficácia
# Usamos 70% para treino e 30% para teste.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [87]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 236597 entries, 728084 to 337125
Data columns (total 28 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   sinal_ears_atend       236597 non-null  int64  
 1   sinal_evi_ivas         236597 non-null  int64  
 2   EWS_ISF                236597 non-null  int64  
 3   EWS_LOF                236597 non-null  int64  
 4   EWS_OCSVM              236597 non-null  int64  
 5   EWS_COPOD              236597 non-null  int64  
 6   EWS_Rt                 236597 non-null  int64  
 7   sinal_ears_atend_lag1  236597 non-null  float64
 8   sinal_ears_atend_lag2  236597 non-null  float64
 9   sinal_ears_atend_lag3  236597 non-null  float64
 10  sinal_evi_ivas_lag1    236597 non-null  float64
 11  sinal_evi_ivas_lag2    236597 non-null  float64
 12  sinal_evi_ivas_lag3    236597 non-null  float64
 13  EWS_ISF_lag1           236597 non-null  float64
 14  EWS_ISF_lag2           236597 non-nu

# --- 3. Emsemble XGBoost (XGBClassifier) ---



## --- Treinamento do classificador com XGBoost  ---


In [88]:
# Abordagem Binária: Calculando o peso automaticamente com scale_pos_weight=counter 
counter = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
xgb_model = xgb.XGBClassifier(
    n_estimators=100,      # Número de árvores (modelos) a serem construídos
    learning_rate=0.1,     # Taxa de aprendizado
    max_depth=3,           # Profundidade máxima de cada árvore para evitar overfitting
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=counter, # controls the balance of positive and negative weights
    max_delta_step=1 # limits the maximum change in the predictions, preventing the model from giving too much importance to the minority class
)  
xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=1,
              max_depth=3, max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

# Avaliação / Métricas

In [89]:
# ==============================================================================
# 1. PREPARAÇÃO DOS DADOS (ORDENAÇÃO TEMPORAL)
# ==============================================================================

# Recria o dataframe com as informações necessárias
df_eval = df.loc[X_test.index].copy()
df_eval['y_true'] = y_test

# configuracao do modelo a ser avaliado 
#Binario = xgb_model
df_eval['y_prob'] = xgb_model.predict_proba(X_test)[:, 1]

# Ordena cronologicamente (ajuste 'co_ibge7' se necessário)
if 'co_ibge7' in df_eval.columns:
    df_eval = df_eval.sort_values(by=['co_ibge7', 'year_week'])
else:
    df_eval = df_eval.sort_values(by=['year_week'])

In [90]:
# ==============================================================================
# 2. FUNÇÃO DE CÁLCULO DE MÉTRICAS 
# ==============================================================================

def calcular_metricas_por_threshold_evento(df, threshold, window_weeks=3):
    # Aplica o corte
    y_pred = (df['y_prob'] >= threshold).astype(int)
    
    # --- A. Métricas de Classificação (Semana a Semana) ---
    tn, fp, fn, tp = confusion_matrix(df['y_true'], y_pred).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # MÉTRICAS DE MACHINE LEARNING:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    
    
    # --- B. Métricas Epidemiológicas (Por Evento de Surto) ---
    # Identifica onde os surtos começam (transição 0 -> 1 no gabarito)
    y_true_series = pd.Series(df['y_true'].values)
    y_pred_series = pd.Series(y_pred.values)
    
    outbreak_starts = y_true_series[(y_true_series.diff() == 1) & (y_true_series == 1)].index
    total_outbreaks = len(outbreak_starts)
    
    early, timely, missed = 0, 0, 0
    
    for start in outbreak_starts:
        # Define janela de antecipação (ex: 3 semanas antes)
        w_start = max(0, start - window_weeks)
        
        # Verifica se houve alerta na janela anterior
        if y_pred_series[w_start:start].sum() > 0:
            early += 1
        # Verifica se houve alerta no dia
        elif y_pred_series[start] == 1:
            timely += 1
        # Se não houve nenhum dos dois
        else:
            missed += 1

    # --- C. Métricas de Falsos Positivos por Evento ---
    # Cria uma máscara onde é 1 apenas nas semanas que são FP (y_true=0 e y_pred=1)
    fp_mask = ((y_true_series == 0) & (y_pred_series == 1)).astype(int)
    
    # Conta quantas vezes a máscara passa de 0 para 1 (início de um evento de alarme falso)
    fp_events = (fp_mask.diff() == 1).sum()
    
    # Tratamento: se a primeira linha de todas do dataset já for um alarme falso, 
    # o diff() dá NaN, então adicionamos 1 manualmente para não perder essa contagem.
    if not fp_mask.empty and fp_mask.iloc[0] == 1:
        fp_events += 1
            
    return {
        "Threshold": threshold,
        # Contagens Epidemiológicas
        "Total Surtos (Eventos)": total_outbreaks,
        "Early Detection (Count)": early,
        "Timely Detection (Count)": timely,
        "Missed Outbreaks (Count)": missed,
        # Percentuais Epidemiológicos
        "Early Detection (%)": f"{early/total_outbreaks:.2%}" if total_outbreaks > 0 else "0.00%",
        "Timely Detection (%)": f"{timely/total_outbreaks:.2%}" if total_outbreaks > 0 else "0.00%",
        "Missed Outbreaks (%)": f"{missed/total_outbreaks:.2%}" if total_outbreaks > 0 else "0.00%",
        "Cobertura Total (%)": f"{(early+timely)/total_outbreaks:.2%}" if total_outbreaks > 0 else "0.00%",
        # Métricas de Machine Learning (comumente usadas em Computação)
        "Accuracy": f"{accuracy:.2%}",
        "Precision (PPV)": f"{precision:.2%}",
        "Sensitivity (Recall)": f"{sensitivity:.2%}",
        "Specificity": f"{specificity:.2%}",
        "F1-Score": f"{f1_score:.4f}", # se quiser alterar para % usar: f"{f1_score:.2%}"
        
        # Métricas de Classificação Clássicas
        "Sensitivity (Recall)": f"{sensitivity:.2%}",
        "Specificity": f"{specificity:.2%}",
        # Comparativo de Falsos Positivos (Semanas x Eventos)
        "Alarmes Falsos (Semanas/Linhas)": fp,
        "Eventos de Alarme Falso (FP Events)": int(fp_events)
    }

In [91]:
# ==============================================================================
# 3. GERAR RELATÓRIO COMPARATIVO
# ==============================================================================

# Calcula para os dois cenários
metrics_050 = calcular_metricas_por_threshold_evento(df_eval, 0.50)
#metrics_065 = calcular_metricas_por_threshold_evento(df_eval, 0.65)

# Cria DataFrame para exibição lado a lado
df_comparativo = pd.DataFrame([metrics_050]) # ,metrics_065
df_comparativo = df_comparativo.set_index("Threshold").T # Transpõe para ficar mais legível

print("=== RELATÓRIO MÉTRICAS ===")
print(df_comparativo)

=== RELATÓRIO MÉTRICAS ===
Threshold                               0.5
Total Surtos (Eventos)                29163
Early Detection (Count)               17305
Timely Detection (Count)               5836
Missed Outbreaks (Count)               6022
Early Detection (%)                  59.34%
Timely Detection (%)                 20.01%
Missed Outbreaks (%)                 20.65%
Cobertura Total (%)                  79.35%
Accuracy                             79.37%
Precision (PPV)                      43.53%
Sensitivity (Recall)                 61.66%
Specificity                          83.11%
F1-Score                             0.5103
Alarmes Falsos (Semanas/Linhas)       32993
Eventos de Alarme Falso (FP Events)   25413


# --- Produção - inferência/aplicação do modelo ---

In [92]:
df_com_lags.info()

<class 'pandas.core.frame.DataFrame'>
Index: 788655 entries, 0 to 788654
Data columns (total 61 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   co_ibge                           788655 non-null  int32  
 1   epiweek                           788655 non-null  int32  
 2   year                              788655 non-null  float64
 3   year_week                         788655 non-null  object 
 4   week                              788655 non-null  object 
 5   atend_ivas                        788655 non-null  int32  
 6   atend_totais                      788655 non-null  int32  
 7   mem_surge_01_correct_with_consec  788655 non-null  int32  
 8   warning_final_mem_surge_01        788655 non-null  int32  
 9   sinal_ears_atend                  788655 non-null  int64  
 10  sinal_evi_ivas                    788655 non-null  int64  
 11  EWS_ISF                           788655 non-null  int64 

In [93]:
# ==============================================================================
# APLICAÇÃO DO MODELO EM PRODUÇÃO (artigo)
# Semana 2022-42 até 2025-32
# ==============================================================================

print("Preparando dados para produção...")

# 1. DEFINIR O DATASET DE PRODUÇÃO
df_prod = df_com_lags[
    (df_com_lags['year_week'] >= '2022-42') & 
    (df_com_lags['year_week'] <= '2025-32')
].copy()

print(f"Dados filtrados: {len(df_prod)} semanas/registros encontrados a partir de 2022-42.")

Preparando dados para produção...
Dados filtrados: 788655 semanas/registros encontrados a partir de 2022-42.


In [94]:
#=============================================================================
# rodar para o dado completo
#=============================================================================

#df_prod = df_com_lags

In [95]:
# 2. SEPARAR AS FEATURES DE ENTRADA DO  MODELO
# !!!!!!!!!EXATAMENTE a mesma e na MESMA ORDEM de treino do modelo

colunas_features = base_models_columns #configurado anteriormente acima
X_prod = df_prod[colunas_features]

In [96]:
# 3. VERIFICAÇÃO DE DADOS FALTANTES (Nulos)
# Se os dados não tiverem o sinal dos modelos de entrada, o modelo não pode prever.
if X_prod.isnull().values.any():
    print("Existem dados faltantes (NaN) nas features.")

In [97]:
# 4. FAZER AS PREDIÇÕES
print("Gerando probabilidades com o XGBoost...")
# Pega a probabilidade da classe 1 (Surto)
df_prod['prob_ensemble_xgb'] = xgb_model.predict_proba(X_prod)[:, 1]

Gerando probabilidades com o XGBoost...


In [98]:
# 5. APLICAR O LIMIAR DE ALERTA (THRESHOLDS)
# gerar alertas para 0.50 (definido em reunião para todos os modelos)
df_prod['signal_ensemble_xgb50'] = (df_prod['prob_ensemble_xgb'] >= 0.50).astype(int)

print("Predições concluídas com sucesso!")
print("-" * 60)

# 6. VISUALIZAR OS RESULTADOS MAIS RECENTES
colunas_visualizacao = ['year_week', 'prob_ensemble_xgb', 'signal_ensemble_xgb50'] #'signal_ensemble_xgb65'

display(df_prod[colunas_visualizacao].tail(10)) # Mostra as 10 últimas semanas preditas

Predições concluídas com sucesso!
------------------------------------------------------------


,year_week,prob_ensemble_xgb,signal_ensemble_xgb50
312668,2025-32,0.278549,0
739262,2025-32,0.278549,0
49391,2025-32,0.278549,0
577121,2025-32,0.278549,0
211532,2025-32,0.278549,0
671348,2025-32,0.278549,0
117305,2025-32,0.278549,0
378230,2025-32,0.755550,1
147734,2025-32,0.348766,0
788654,2025-32,0.354158,0


In [99]:
df_prod.info()

<class 'pandas.core.frame.DataFrame'>
Index: 788655 entries, 0 to 788654
Data columns (total 63 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   co_ibge                           788655 non-null  int32  
 1   epiweek                           788655 non-null  int32  
 2   year                              788655 non-null  float64
 3   year_week                         788655 non-null  object 
 4   week                              788655 non-null  object 
 5   atend_ivas                        788655 non-null  int32  
 6   atend_totais                      788655 non-null  int32  
 7   mem_surge_01_correct_with_consec  788655 non-null  int32  
 8   warning_final_mem_surge_01        788655 non-null  int32  
 9   sinal_ears_atend                  788655 non-null  int64  
 10  sinal_evi_ivas                    788655 non-null  int64  
 11  EWS_ISF                           788655 non-null  int64 

In [100]:
# ==============================================================================
# EXPORTANDO RESULTADOS DE PRODUÇÃO PARA PARQUET
# ==============================================================================
df_export = df_prod[['co_ibge',
 'epiweek',
 'year',
 'year_week',
 'week',
 'atend_ivas',
 'atend_totais',
 'mem_surge_01_correct_with_consec',
 'warning_final_mem_surge_01',
 'sinal_ears_atend',
 'sinal_evi_ivas',
 'EWS_ISF',
 'EWS_LOF',
 'EWS_OCSVM',
 'EWS_COPOD',
 'EWS_Rt',
 'prob_ensemble_xgb', 
 'signal_ensemble_xgb50']]

nome_arquivo_parquet = '/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/resultado_predicoes_producao_xgb.parquet'

# Salvando o dataframe
df_export.to_parquet(nome_arquivo_parquet, engine='pyarrow', index=False)
    
print("-" * 50)
print(f"Arquivo Parquet salvo com sucesso: {nome_arquivo_parquet}")
print(f"Total de linhas salvas: {len(df_export)}")
print("-" * 50)


--------------------------------------------------
Arquivo Parquet salvo com sucesso: /opt/storage/shared/aesop/aesop_shared/ensamble_modelling/resultado_predicoes_producao_xgb.parquet
Total de linhas salvas: 788655
--------------------------------------------------


# Verificar saída modelo

In [101]:
df_prod = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/resultado_predicoes_producao_xgb.parquet')
print(f"Tamanho do DataFrame: {df_prod.shape}")
print(f"Total de linhas: {len(df_prod)}")
print("\nVerificando dados faltantes (NaN):")
columns = ['prob_ensemble_xgb', 'signal_ensemble_xgb50']

for col in columns:
    print(f"Valores NaN na coluna '{col}' ")
    print(df_prod[col].isna().sum())

Tamanho do DataFrame: (788655, 18)
Total de linhas: 788655

Verificando dados faltantes (NaN):
Valores NaN na coluna 'prob_ensemble_xgb' 
0
Valores NaN na coluna 'signal_ensemble_xgb50' 
0


In [102]:
df_prod.columns.tolist()

['co_ibge',
 'epiweek',
 'year',
 'year_week',
 'week',
 'atend_ivas',
 'atend_totais',
 'mem_surge_01_correct_with_consec',
 'warning_final_mem_surge_01',
 'sinal_ears_atend',
 'sinal_evi_ivas',
 'EWS_ISF',
 'EWS_LOF',
 'EWS_OCSVM',
 'EWS_COPOD',
 'EWS_Rt',
 'prob_ensemble_xgb',
 'signal_ensemble_xgb50']

In [103]:
colunas_esperadas_pelo_modelo

array(['sinal_ears_atend', 'sinal_evi_ivas', 'EWS_ISF', 'EWS_LOF',
       'EWS_OCSVM', 'EWS_COPOD', 'EWS_Rt', 'sinal_ears_atend_lag1',
       'sinal_ears_atend_lag2', 'sinal_ears_atend_lag3',
       'sinal_evi_ivas_lag1', 'sinal_evi_ivas_lag2',
       'sinal_evi_ivas_lag3', 'EWS_ISF_lag1', 'EWS_ISF_lag2',
       'EWS_ISF_lag3', 'EWS_LOF_lag1', 'EWS_LOF_lag2', 'EWS_LOF_lag3',
       'EWS_OCSVM_lag1', 'EWS_OCSVM_lag2', 'EWS_OCSVM_lag3',
       'EWS_COPOD_lag1', 'EWS_COPOD_lag2', 'EWS_COPOD_lag3',
       'EWS_Rt_lag1', 'EWS_Rt_lag2', 'EWS_Rt_lag3'], dtype='<U21')

In [104]:
# ==============================================================================
# AVALIAÇÃO FINAL NO DATASET DE PRODUÇÃO
# ==============================================================================

# 1. Prepare as Features (X) e o Gabarito (y) da produção
colunas_esperadas_pelo_modelo = xgb_model.feature_names_in_

X_prod = df_prod[colunas_esperadas_pelo_modelo].copy() # o df_prod não tem as colunas 

y_prod_true = df_prod[ 'mem_surge_01_correct_with_consec']#'warning_final_mem_surge_01'] 

# 2. Gere as probabilidades com o modelo treinado
print("Gerando predições para o período de produção...")
probabilidades_prod = xgb_model.predict_proba(X_prod)[:, 1]

# 3. Cria um DataFrame temporário só com o que a função precisa ler
df_avaliacao_prod = pd.DataFrame({
    'y_true': y_prod_true.values,
    'y_prob': probabilidades_prod
})

# 4. Roda função de métricas
print(f"\n--- Resultados de Produção")
resultados_prod = calcular_metricas_por_threshold_evento(df_avaliacao_prod, threshold=0.50)

# 5. Imprime os resultados formatados
for metrica, valor in resultados_prod.items():
    print(f"{metrica.ljust(35)}: {valor}")

KeyError: "['sinal_ears_atend_lag1', 'sinal_ears_atend_lag2', 'sinal_ears_atend_lag3', 'sinal_evi_ivas_lag1', 'sinal_evi_ivas_lag2', 'sinal_evi_ivas_lag3', 'EWS_ISF_lag1', 'EWS_ISF_lag2', 'EWS_ISF_lag3', 'EWS_LOF_lag1', 'EWS_LOF_lag2', 'EWS_LOF_lag3', 'EWS_OCSVM_lag1', 'EWS_OCSVM_lag2', 'EWS_OCSVM_lag3', 'EWS_COPOD_lag1', 'EWS_COPOD_lag2', 'EWS_COPOD_lag3', 'EWS_Rt_lag1', 'EWS_Rt_lag2', 'EWS_Rt_lag3'] not in index"